HACKATHON COLAS — Équipe DataBeton
Classification du contenu de camions de carrière

Contexte :
Colas est un acteur majeur du granulat, représentant 15%
du marché mondial en volume. L'objectif est de construire
un modèle capable de reconnaître le contenu d'un camion
à partir d'une image prise sur site de carrière.

Classes officielles à prédire :
  CAMION_VIDE       — camion vide, bâché, ou sans contenu visible
  EBALBRUT          — matériau brut concassé sombre
  EBETDPRETFER_BAT  — grands blocs / béton
  S0_315RGR1        — agrégat fin à moyen gris
  S0_6R             — matériau sableux fin
  S30_80R           — gravier grossier
  SGTLH0_3153VR     — agrégat compact fin/moyen sombre

Ce que ce notebook fait réellement :
  1. Exploration du dataset et identification du bruit de labels
  2. Correction manuelle : 266 camions vides détectés et
     relabellisés, 28 images invalides ou ambiguës supprimées
  3. Entraînement EfficientNet-B0 par Transfer Learning
     avec AdamW et Label Smoothing
  4. Évaluation sur un test interne holdout (70/15/15)
     pour détecter l'overfitting
  5. Expérience séparée sur le biais de site EBE/SBE
  6. Inférence finale sur les 282 images de test

RÉSUMÉ DE LA MÉTHODOLOGIE

On a traité ce sujet comme un problème de vision industrielle
opérationnelle, pas uniquement comme un classifieur à 7 classes.

Dans un contexte carrière, les images peuvent contenir des
camions vides, des camions bâchés, des routes sans camion,
des vues partielles ou des images de nuit. Forcer une classe
produit sur chaque image créerait de fausses informations
opérationnelles pour Colas.

Notre démarche en 4 étapes :

  1. EXPLORATION ET NETTOYAGE MANUEL
     Le CSV original contient 1 seul CAMION_VIDE labellisé
     sur 2101 images. On a inspecté visuellement toutes les
     images pour détecter les camions vides cachés dans les
     classes produits, identifier les images invalides et
     corriger le bruit de labels.

  2. ENTRAÎNEMENT EfficientNet-B0
     Transfer Learning sur un modèle pré-entraîné ImageNet,
     avec AdamW et Label Smoothing pour réduire l'overfitting
     et améliorer la robustesse sur les labels bruités.
     Évalué sur un split 70/15/15 avec test interne.

  3. ANALYSE DU BIAIS DE SITE
     On a détecté que certaines classes viennent quasi-
     exclusivement d'un seul site (EBE ou SBE). On a testé
     une approche séparée pour mesurer et réduire ce biais.

  4. INFÉRENCE ET SOUMISSION
     Application du modèle sur les 282 images de test.
     Les images sans contenu visible sont assignées à
     CAMION_VIDE, seule classe officielle disponible.

Étape 0 — Imports

On commence par importer toutes les librairies nécessaires :
- pandas : pour lire et manipuler le fichier CSV des labels
- PIL (Pillow) : pour ouvrir les images
- torch / torchvision : le framework de deep learning
- sklearn : pour encoder les labels et découper le dataset

In [0]:
import os
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

print("Librairies importées avec succès.")

Librairies importées avec succès.


Étape 1 — Exploration du dataset

On charge le fichier labels.csv et on vérifie :
- le nombre total d'images et de classes
- la distribution des classes
- la présence de doublons ou de labels inattendus

Observation clé :
Les 6 classes produit sont presque équilibrées (~350 images
chacune), MAIS CAMION_VIDE n'a qu'UN SEUL exemple officiel
sur 2101 images au total.

Cela pose deux problèmes immédiats :

  Problème 1 — Apprentissage impossible
  Avec 1 seul exemple, le modèle ne peut pas apprendre
  une représentation robuste des camions vides. Il les
  confondrait systématiquement avec des classes produits.

  Problème 2 — Labels bruités
  En inspectant visuellement les images, on constate que
  de nombreux camions vides sont cachés dans les classes
  produits avec un mauvais label. C'est l'obstacle
  volontaire mentionné dans le sujet.

  Ces deux problèmes justifient la correction manuelle des labels décrite à l'étape suivante.

In [0]:
BASE_PATH    = "/Volumes/efrei_estp_2026/input_data/quarries-trucks-dataset"
HACKATON_DIR = f"{BASE_PATH}/hackaton"
TEST_DIR     = f"{BASE_PATH}/test"
LABELS_CSV   = f"{BASE_PATH}/labels.csv"

df = pd.read_csv(LABELS_CSV)

print(f"Nombre total d'images : {len(df)}")
print(f"\nAperçu du fichier CSV :")
print(df.head())
print(f"\nNombre d'images par classe :")
print(df["label"].value_counts())

Nombre total d'images : 2101

Aperçu du fichier CSV :
                                               image        label
0                                    CAMION-VIDE.png  CAMION_VIDE
1  BL-LC24026888C-EBE-f441361b-1ecc-447d-b6f1-8b2...     EBALBRUT
2  BL-LC24027000C-EBE-48b29ae1-1902-4ed3-b59d-328...     EBALBRUT
3  BL-LC24030199C-EBE-33c5c139-1161-4e05-8f92-6e6...     EBALBRUT
4  BL-LC24030589C-EBE-ddb4aea7-60c4-4b6c-a87f-c8b...     EBALBRUT

Nombre d'images par classe :
EBALBRUT            350
EBETDPRETFER_BAT    350
S0_315RGR1          350
S0_6R               350
S30_80R             350
SGTLH0_3153VR       350
CAMION_VIDE           1
Name: label, dtype: int64


Étape 1.2 — Correction manuelle des labels

Suite à l'exploration visuelle du dataset, on a créé
3 listes pour corriger le bruit de labels :

1. manual_empty_truck_images (266 images)
   Camions clairement vides, camions bâchés, et routes
   sans camion visible → relabellisés en CAMION_VIDE.

   Choix : on inclut les camions bâchés et routes vides
   car ces cas existent aussi dans le jeu de test. Si on
   les supprimait du train, le modèle ne les reconnaîtrait
   pas à l'inférence.

   Limite connue : une route vide n'est pas strictement
   un camion vide. Ce choix peut légèrement dégrader la
   précision sur CAMION_VIDE.

2. manual_invalid_images (17 images)
   Images corrompues, inutilisables, ou sans aucun contenu
   exploitable → supprimées du dataset.

3. manual_uncertain_images (11 images)
   Vues trop partielles ou contenu vraiment trop ambigu
   pour être exploitable → supprimées du dataset.

Résultat attendu après nettoyage :
  CAMION_VIDE passe de 1 à 266 exemples
  Dataset total : ~2073 images (vs 2101 avant)
  Distribution restante : équilibrée entre ~283 et 335

Truck visible but clearly empty.

In [0]:
manual_empty_truck_images = [
    'BL-LC25087675C-EBE-ee24082e-3bf4-4c4a-a540-424b917c5d14.JPG',
    'BL-LC25075316C-EBE-153bb68d-a593-47f4-b428-ac55852b79b9.JPG',
    'BL-LC25074565C-EBE-213acaa8-3b5b-4864-8142-f7caa895f31e.JPG',
    'BL-LC25074378C-EBE-da509cdc-0021-4229-ab50-ccd7724bfd56.JPG',
    'BL-LC25074374C-EBE-23bd2813-57c5-4923-a081-76c602d75042.JPG',
    'BL-LC25066629C-EBE-dda18bf7-6fcb-41ff-b4b1-4bfd55ee9e2a.JPG',
    'BL-LC25066240C-EBE-011bdde4-3609-4407-8b51-1901e03fac63.JPG',
    'BL-LC25063172C-EBE-c075c0bf-55f6-4b3f-a04c-bcec47baad29.JPG',
    'BL-LC25046886C-EBE-1366d311-d874-4d50-bc99-062743eb1fbe.JPG',
    'BL-LC25038503C-EBE-1263b73b-927c-457e-8828-53b119dff7f2.JPG',
    'BL-LC25033294C-SBE-e0872735-32f0-4f78-a812-acce71b9b43f.JPG',
    'BL-LC25025848C-EBE-0e6207fd-c001-4e51-9f9b-70734bca2937.JPG',
    'BL-LC25014207C-EBE-8ea6541f-f4ee-4344-80c2-8882846f771e.JPG',
    'BL-LC25014192C-EBE-9c200522-25aa-4c81-85ca-32cfd2aff3af.JPG',
    'BL-LC25013937C-EBE-77ec5ad3-5099-45c7-8924-cc4c459b7808.JPG',
    'BL-LC24122777C-EBE-ac4f2247-20cf-4c24-b146-1e8c89c2be22.JPG',
    'BL-LC24121557C-SBE-0fcf9f2c-b69c-459c-81e6-5a2b93ef13d1.JPG',
    'BL-LC24121481C-SBE-e0eacd48-70e8-47ac-ae81-f80bd9fe2e36.JPG',
    'BL-LC24110706C-SBE-ddb292da-15e6-4db8-8be4-7bc55386eabb.JPG',
    'BL-LC24108923C-EBE-5f056be6-64fe-4e35-8ba2-83fc1a2d7f5c.JPG',
    'BL-LC24108010C-EBE-e7aba8e0-c7a3-43a3-b267-ec7224a2785f.JPG',
    'BL-LC24105074C-EBE-55727c44-86ed-43d3-ab19-fdc1f4c047b3.JPG',
    'BL-LC24103792C-SBE-6e997896-5b24-4206-8d3d-1f363cf96fc6.JPG',
    'BL-LC24103766C-SBE-4039ffe5-bd64-41c3-bb66-19f79da7fe53.JPG',
    'BL-LC24085747C-EBE-4ac825d0-7699-4d0f-a5ee-b9728c6eaa31.JPG',
    'BL-LC24083466C-SBE-da422c40-7689-45ea-ab2b-1efd121ceebf.JPG',
    'BL-LC24081313C-EBE-a8a5d647-346a-48de-9767-bde7ebbb8dd7.JPG',
    'BL-LC24078755C-EBE-cb7407a2-fab9-4cc7-b0db-9cdfa7f90f58.JPG',
    'BL-LC24063607C-EBE-f3e4927e-b2a7-4d62-8610-84c048d51cca.JPG',
    'BL-LC24063590C-EBE-61bbff98-802f-44db-8400-6c9c28b54a2a.JPG',
    'BL-LC24062227C-EBE-705dc166-beb0-4233-8bdc-b402f861dff3.JPG',
    'BL-LC24060887C-EBE-4f27088f-4d12-4b9f-90df-53310abeb185.JPG',
    'BL-LC24056552C-EBE-5aa44a16-f1e8-4b5f-9d42-22294d5194e2.JPG',
    'BL-LC24050793C-SBE-40b6b4ed-44e1-45f1-bd1d-29adb9ffeaa7.JPG',
    'BL-LC24050662C-SBE-69cb7864-a5a0-44fb-ac5f-54003939743c.JPG',
    'BL-LC24046881C-SBE-f175b838-c114-4ce5-9fad-78805b833dc9.JPG',
    'BL-LC24041003C-SBE-01ca1bce-84cf-4aab-9069-57c8c318b604.JPG',
    'BL-LC24042429C-EBE-222798b0-6196-4130-8346-90befd61c0e7.JPG',
    'BL-LC25052240C-SBE-6e9d52a5-0ecc-4b25-add9-b509d3b3fcc9.JPG',
    'BL-LC25052511C-EBE-87dc581e-baba-4124-9226-2ad2185866b9.JPG',
    'BL-LC25052512C-EBE-e27663b9-137c-4e31-a82a-480f07032ec1.JPG',
    'BL-LC25052514C-EBE-df4272b2-22fd-420a-9d74-04a35783c441.JPG',
    'BL-LC25055287C-EBE-6d456834-d947-47c7-867c-ba2c0715affb.JPG',
    'BL-LC25055764C-EBE-fd37a9e7-60fd-4812-80e0-da1685887296.JPG',
    'BL-LC25056412C-EBE-2267631f-965b-4840-8d60-935f722c6411.JPG',
    'BL-LC25076625C-SBE-0dde1404-195c-4e7e-b637-e762be28afa9.JPG',
    'BL-LC25076688C-EBE-f4aa068a-40df-48a4-9d5f-31f3052d10a5.JPG',
    'BL-LC25078302C-EBE-3b6a9068-78ab-4d56-8c71-d9dae42bea41.JPG',
    'BL-LC25080002C-EBE-87e3ee22-33e6-489a-8acd-611999ea52e8.JPG',
    'BL-LC25080084C-EBE-cf175b3c-038f-49a2-942e-e60fb84547c8.JPG',
    'BL-LC25080443C-EBE-5caf6476-cd5e-48fa-bd6e-408ec7d8ab8c.JPG',
    'BL-LC25080950C-EBE-c7f30c3e-702a-4d42-bd7d-c11b9dcf389b.JPG',
    'BL-LC25081065C-EBE-c5f5c949-7962-470b-9d8f-053f108549cb.JPG',
    'BL-LC25082260C-EBE-439dff63-8479-4fb4-a8b5-c1324c67fa4a.JPG',
    'BL-LC25092105C-SBE-626d3309-1401-4566-ab44-cfe1bd9a91a7.JPG',
    'BL-LC25092515C-EBE-8a749716-57d2-4b93-8fba-5b5b88ca9343.JPG',
    'BL-LC25093026C-EBE-40bd209b-f31d-4bf3-ac73-630e9bd8766b.JPG',
    'BL-LC25094001C-SBE-c776d0ef-6c71-4471-b7bc-ee97a95594eb.JPG',
    'BL-LC25094138C-SBE-0b356d8a-f110-4ff6-afd5-15c2cad7b8e6.JPG',
    'BL-LC25095106C-EBE-dba569cc-aadd-4342-8c0c-58befbcaf908.JPG',
    'BL-LC25095853C-SBE-20801715-8515-42a0-8cd5-3338bdadce39.JPG',
    'BL-LC25096569C-SBE-4f412517-d811-48da-913b-e3ac63a29904.JPG',
    'BL-LC25096728C-EBE-3047cfdb-1580-4c31-a862-e94f09b0fcad.JPG',
    'BL-LC25097372C-SBE-2964b0f7-4c90-44f3-a538-e4699274de54.JPG',
    'BL-LC25097484C-EBE-6ee4d67a-fc16-4ad0-afb1-10be1796ac87.JPG',
    'BL-LC25098002C-EBE-e580f815-ad1d-4852-b5c7-410016f25a3b.JPG',
    'BL-LC25098320C-EBE-4f8f7142-df1c-4677-a899-900d163736b6.JPG',
    'BL-LC25099847C-EBE-56263c36-ac45-4fe8-a4f2-6b2b2e231b2b.JPG',
    'BL-LC25100530C-EBE-de9a6ca4-b0cb-4d13-937b-47557621ba1b.JPG',
    'BL-LC25100615C-EBE-08073151-ad73-458e-ad49-30972db98242.JPG',
    'BL-LC25101237C-EBE-5e317216-234b-44c5-9eee-8b0b3d030140.JPG',
    'BL-LC25101313C-EBE-df99c015-9296-4872-bbe6-ae9341ea7c68.JPG',
    'BL-LC25101463C-EBE-ce870d12-8bbb-4964-857f-7bf38f403349.JPG',
    'BL-LC25101471C-EBE-09c3510d-23ac-4bbf-b27e-420340458cb7.JPG',
    'BL-LC25101871C-EBE-6aa5d131-491e-4c4d-b24f-fdaffe183901.JPG',
    'BL-LC25102413C-EBE-7556ad70-79d7-4240-a331-1a53e6199b37.JPG',
    'BL-LC25103461C-EBE-5bda49a9-cea3-4194-a019-77a53c4b714a.JPG',
    'BL-LC25103468C-EBE-addd0cd1-68e8-49df-bca7-bf1ab38a987e.JPG',
    'BL-LC25105171C-EBE-ef08e99d-b3bb-4917-aad0-7007d3768513.JPG',
    'BL-LC25105336C-EBE-834bf77d-2190-4e88-9d77-7694a59a49c3.JPG',
    'BL-LC25105386C-EBE-01d6a567-31f4-499b-b504-276d5285d5c7.JPG',
    'BL-LC25106109C-EBE-411337e6-a717-46b7-89bb-68821aecf10f.JPG',
    'BL-LC25107140C-SBE-e1aec48b-837d-448c-bc54-a42cd77cd9b3.JPG',
    'BL-LC25107203C-EBE-cc80be4a-c784-4acd-87e9-ed4578cdf979.JPG',
    'BL-LC25107563C-SBE-a217dc1c-7925-4ef3-80de-63f486492e41.JPG',
    'BL-LC25109694C-EBE-4162e577-c363-4f70-92b6-82a5187382a4.JPG',
    'BL-LC25111525C-EBE-f8751182-a1a2-4bd8-bd7c-86c8fcb585f2.JPG',
    'BL-LC25112015C-EBE-d35bad89-2c66-4946-9102-900039503d05.JPG',
    'BL-LC25112437C-SBE-ce4784dd-943b-43f3-be5f-51d5923b7474.JPG',
    'BL-LC25112748C-EBE-84933fe4-8949-4c14-8802-b8745a287aed.JPG',
    'BL-LC25114617C-EBE-6241139e-c7ba-49b2-9a29-31b37d3ab826.JPG',
    'BL-LC25114666C-EBE-b5adc895-f4fe-43f0-8278-83485b84fc5c.JPG',
    'BL-LC25114933C-SBE-ac83473d-48d9-4bea-a7c0-fee5d3edae8c.JPG',
    'BL-LC25115468C-SBE-5dbbdbf6-55c7-459c-9f71-065675fe757a.JPG',
    'BL-LC25115958C-EBE-ae0eff52-bf84-447f-aacf-ceb83c5d0d87.JPG',
    'BL-LC25116078C-EBE-5c828efa-a75b-4d54-ada0-c5902c373837.JPG',
    'BL-LC25116177C-SBE-29a0ad90-7d2c-48a4-b729-34275d83fb2b.JPG',
    'BL-LC25116214C-EBE-112c2196-93a6-4230-a456-57de6dff5833.JPG',
    'BL-LC25121788C-EBE-7216f0c6-54d7-438c-bda5-6badaaa62244.JPG',
    'BL-LC25120748C-EBE-042b553e-048a-4827-b7bb-3cd79b2436d3.JPG',
    'BL-LC25120762C-EBE-2b073526-0114-42ff-a4a9-058dcff6b9f1.JPG',
    'BL-LC25122194C-EBE-43575080-6441-46e1-9bb0-5a21b754b68d.JPG',
    'BL-LC25123074C-EBE-de7aab57-a568-4b86-a4e2-8e66843ad714.JPG',
    'BL-LC25123511C-EBE-6711cda8-e670-4128-abf9-c6bc872109df.JPG',
    'BL-LC25123591C-SBE-69ae60d2-a991-4904-ac72-6a990d57e185.JPG',
    'BL-LC25124243C-SBE-55ba63ed-b575-4feb-a740-7321aa8e2d14.JPG',
    'BL-LC25124701C-SBE-b82960d7-6baf-47f4-b35c-57a45c8116d3.JPG',
    'BL-LC25124848C-SBE-adf72ee5-d9c2-454d-be64-7728d323cae8.JPG',
    'BL-LC25125233C-SBE-bf71d228-2487-48de-b924-62dd5d067bae.JPG',
    'BL-LC25125374C-SBE-cb5e9cc5-5b22-47c3-8c27-8b21a21c0643.JPG',
    'BL-LC25125465C-SBE-770d819a-9563-42a4-ac54-8f55efeb7b92.JPG',
    'BL-LC25125689C-EBE-aa7f929e-34ab-4bed-8d40-381c1451a21f.JPG',
    'BL-LC25125710C-SBE-d8afc337-ea4b-4b5f-843c-b7c87a17850f.JPG',
    'BL-LC25125744C-SBE-74d0cc56-17f8-4a75-bfba-25ac30f0497b.JPG',
    'BL-LC25125782C-SBE-0abbfb93-c83c-44ea-b387-8c92f19ddc05.JPG',
    'BL-LC26010462C-SBE-23638968-3c9a-42f6-9071-fab97e837d2a.JPG',
    'BL-LC26010604C-SBE-de67a0f3-59c7-4636-8b29-8a1fd19864ce.JPG',
    'BL-LC26013005C-SBE-369126a4-395e-414a-95d9-b85f91f884d3.JPG',
    'BL-LC26013399C-SBE-b666cc69-b088-48b2-ad5f-a74d3efbbf3e.JPG',
    'BL-LC26013645C-SBE-773ba527-9da2-4843-90aa-c073a33f69d5.JPG',
    'BL-LC26013705C-SBE-259e31e0-df79-418d-b1c5-7cd89d26fa88.JPG',
    'BL-LC26014193C-SBE-6d7d8ec2-8892-4b5f-a549-7c43845c9d9f.JPG',
    'BL-LC26014208C-SBE-e717d43e-7fe2-4b78-b694-135904d47caa.JPG',
    'BL-LC26014965C-SBE-028071fc-ffaa-4f72-85ee-92910d72f541.JPG',
    'BL-LC26014975C-SBE-015ebde4-18b2-4f51-b391-61d88142596e.JPG',
    'BL-LC26015708C-SBE-5d97c2ba-30be-4de6-b8c5-b34d872a5b9c.JPG',
    'BL-LC26020742C-SBE-2f48359f-8958-4347-b6bd-6e437da6194a.JPG',
    'BL-LC26023575C-SBE-a71d7ce1-1fc6-4c31-9369-5880d4ff93c1.JPG',
    'BL-LC26023829C-EBE-9d7a9431-2994-4902-b01a-e565654d4124.JPG',
    'BL-LC26024131C-SBE-10f1ce8d-43b6-4dfa-b700-eea9b83867eb.JPG',
    'BL-LC26024242C-SBE-f1ebdf58-0b9f-4a2f-bce2-acc786cde619.JPG',
    'BL-LC26025898C-EBE-93eeb569-158f-4f94-8486-7bd74b205e91.JPG',
    'BL-LC26030063C-SBE-0bb09f8e-a956-4db4-bd24-38c8f3e22146.JPG',
    'BL-LC26030423C-EBE-771a058c-5ee3-4dba-acc6-8669a7c22ad2.JPG',
    'BL-LC26031321C-EBE-7b6a963d-6cf5-4452-8ea0-6a269e8aef9d.JPG',
    'BL-LC26031456C-EBE-90aed3cc-9cdd-4da2-a3e4-b454c7479694.JPG',
    'BL-LC26032146C-SBE-fd31c8e0-483e-4426-b29b-9f89f6d7d241.JPG',
    'BL-LC26032724C-SBE-36790819-ab7e-4832-97a8-b45631fbd7c4.JPG',
    'BL-LC26032769C-SBE-05c7599f-0799-4048-a4d7-6c43b9348849.JPG',
    'BL-LC26033316C-EBE-905653d9-2e8f-4db6-9f3f-9fe59901ad32.JPG',
    'BL-LC250910581C-EBE-f8e262ce-3c59-4118-be5e-1ff4340a68ba.JPG',
    'BL-LC251010243C-EBE-618f681f-ebcc-4438-8dfa-4eebaed2e4e0.JPG',
    'BL-LC24021203C-SBE-06222883-b188-43cb-9f52-7e5f3dc52c53.JPG',
    'BL-LC24032485C-SBE-060cfe2e-5049-4f7b-bbb5-d16f996af3ae.JPG',
    'BL-LC24032764C-SBE-ea72ee94-83f8-401a-b353-122b1b8fcc57.JPG',
    'BL-LC24041661C-SBE-c82b3c38-8fe9-4c3c-b240-0e415afe217b.JPG',
    'BL-LC24043383C-SBE-c7c7717e-0826-4154-aea9-ff42f4ddbc96.JPG',
    'BL-LC24045339C-SBE-73a4db9f-db2a-4b62-a846-624cc48bf20e.JPG',
    'BL-LC24045487C-SBE-3eecc09b-3566-4bdf-b105-e93fb5f01657.JPG',
    'BL-LC24046100C-SBE-51f55253-1bbb-4935-8218-10c0c1b11df4.JPG',
    'BL-LC24046422C-SBE-f6204446-1ce9-4875-a8cb-d635c4b8e7a7.JPG',
    'BL-LC24046691C-SBE-36029345-844f-418a-b1d5-0b016ab16585.JPG',
    'BL-LC24046908C-SBE-397472b3-eba0-436d-82e0-eff0038eab54.JPG',
    'BL-LC24053389C-SBE-9556d5c1-d46c-4f7d-94f5-4ceae201411d.JPG',
    'BL-LC24053593C-SBE-9c0246a4-e9fa-4072-8e39-2dba154c9ff5.JPG',
    'BL-LC24054558C-SBE-4c06496e-47d0-4caa-9736-a2eb091bacfa.JPG',
    'BL-LC24054688C-SBE-747bebb8-c917-4477-8e22-435e89b26f8d.JPG',
    'BL-LC24055309C-SBE-a0e69c2e-39a2-4213-98c6-a6270769badf.JPG',
    'BL-LC24055760C-EBE-82da89b9-09a9-4f0d-8105-d96e0702e3c6.JPG',
    'BL-LC24055765C-EBE-bf6b2630-c549-4285-991d-50548bdf807d.JPG',
    'BL-LC24055774C-SBE-c1caf989-4a56-4c98-bb3c-8b8b1dbbdf27.JPG',
    'BL-LC24055949C-SBE-181a96dd-d017-4f75-befc-a622eeb60cb8.JPG',
    'BL-LC24056431C-SBE-ffbeeaec-4b39-459b-8bde-6bb7441af7fd.JPG',
    'BL-LC24062119C-SBE-aa68dd1b-b26f-4a4b-aed0-27f8887d53d1.JPG',
    'BL-LC24063259C-SBE-658c2414-b76a-4dd6-b658-c4865fb6db34.JPG',
    'BL-LC24067678C-SBE-3dfacdd6-f1a6-4ae1-b8ed-9bc3abc19238.JPG',
    'BL-LC24071402C-SBE-a562454c-81dd-49e4-b16f-4e8106fe0a02.JPG',
    'BL-LC24072503C-SBE-d5a55dd8-2d03-4fea-b25a-82014030fc8c.JPG',
    'BL-LC24075726C-EBE-86777249-6694-4786-bb33-edf7cd8022fe.JPG',
    'BL-LC24082652C-SBE-85f3f5b5-dea2-4b21-81f0-5b58be98b448.JPG',
    'BL-LC24085237C-SBE-4604ec7d-2069-4b0a-a819-922d90b804c0.JPG',
    'BL-LC24093343C-SBE-be360b57-6458-4436-817f-c25625087254.JPG',
    'BL-LC24102585C-SBE-dd33c069-7435-478c-a269-03479251506b.JPG',
    'BL-LC24107603C-SBE-90247cbb-96ac-4b35-b705-95e0d2b01bf7.JPG',
    'BL-LC24107994C-EBE-5cf5b7bb-a399-4fa2-b9df-6cef21d17480.JPG',
    'BL-LC24112848C-SBE-212fefce-4485-43f4-8fe8-5b0a041a5340.JPG',
    'BL-LC24113056C-EBE-33134a1f-7a8e-4181-8237-d3f22b8cc01a.JPG',
    'BL-LC24114048C-SBE-2b8798b0-cb9e-4be3-a6a8-b622a265651f.JPG',
    'BL-LC24115071C-EBE-6c2e8b6d-b781-428b-9132-a7ed4c45418d.JPG',
    'BL-LC24120015C-EBE-7af32f38-cad2-460c-9609-7407e3ed11ad.JPG',
    'BL-LC24120650C-SBE-d5f49c7f-0940-4802-8ffd-a66959797526.JPG',
    'BL-LC24120990C-SBE-eb56c99b-bd29-40d4-acfe-ec8d3281e76b.JPG',
    'BL-LC24121270C-SBE-004428c5-467a-468d-81b9-74d3f52f9e2f.JPG',
    'BL-LC24122093C-SBE-3486ab57-c81b-4a18-98b3-ea0f74ea2231.JPG',
    'BL-LC25014205C-EBE-830316e5-71eb-4196-8044-d21211fc7cf1.JPG',
    'BL-LC25029641C-SBE-afc6ef83-27e7-4d75-8999-e8f1ce06ec0d.JPG',
    'BL-LC25034984C-SBE-ba4cd1cf-f8d7-4f42-8ddb-6bdaf5657331.JPG',
    'BL-LC25035079C-EBE-d933e45c-7a58-475b-aad6-ef60c2d4f058.JPG',
    'BL-LC25035236C-EBE-13b8e052-f8cd-4441-8e61-27ce9f6f5678.JPG',
    'BL-LC25052901C-EBE-cb1fe4ce-f570-46ba-9a9f-fe3a36d138a2.JPG',
    'BL-LC25061727C-SBE-5facea19-0a1a-4452-abcc-ed116e8ea508.JPG',
    'BL-LC25061869C-SBE-0a337287-409a-4c6e-ac65-e5f841e72d76.JPG',
    'BL-LC25061932C-SBE-cfcdc33f-b50d-4735-ad22-5855a2775430.JPG',
    'BL-LC25065753C-SBE-13e8b156-0807-41fa-88d1-bd109b74ed80.JPG',
    'BL-LC25068006C-EBE-34d1394a-c7df-4713-a359-d2bdd9218b09.JPG',
    'BL-LC25070909C-EBE-2e1ca95b-365c-4112-bdc3-c85994b52fb0.JPG',
    'BL-LC25074006C-SBE-06de6dfa-b59f-4935-9eaf-c0d681f45d04.JPG',
    'BL-LC25074763C-SBE-29fb89ef-54f5-40f2-b2e7-dcee58eaad22.JPG',
    'BL-LC25075289C-SBE-c358c1cd-b89c-4088-92ca-84450a9e44b9.JPG',
    'BL-LC25075873C-EBE-ccfa93d8-1301-4436-84a6-63903626d2a6.JPG',
    'BL-LC25079708C-SBE-d676035c-4209-4606-b765-cb42e902e812.JPG',
    'BL-LC25086993C-EBE-3a13cc45-c13f-45da-91b8-4df36ffe3afe.JPG',
    'BL-LC25087140C-EBE-c24c8ca1-b7c0-453f-9cc2-2c5772568b39.JPG',
    'BL-LC25087452C-SBE-863338d8-cc0c-4247-aa2f-fcca5d057bbe.JPG',
    'BL-LC25087815C-SBE-b484146c-703d-45f6-8db7-429c5a86d0b6.JPG',
    'BL-LC25088292C-SBE-ca9df905-e4bc-4806-ab45-3e279d1532dc.JPG',
    'BL-LC25088617C-SBE-66439200-bb95-4cbc-9da9-820fd4d0c343.JPG',
    'BL-LC25088676C-SBE-e3b047d0-e22b-46df-afe9-626124d0ef82.JPG',
    'BL-LC25090003C-EBE-39ef6611-dca4-4b84-99fc-7bd24d4c381d.JPG',
    'BL-LC25090102C-EBE-8dd12ad4-78c6-4697-a990-2fc981e1aa96.JPG',
    'BL-LC25090885C-SBE-d0f96ee3-2f9b-4668-82fc-80c7f55e8132.JPG',
    'BL-LC25091012C-SBE-549301c8-a7b1-4839-9890-0220ed3989bd.JPG',
    'BL-LC25091488C-SBE-858a1a05-66aa-4c63-a646-fcd71bf2a2ba.JPG',
    'BL-LC25093204C-SBE-a73fcb48-f4c7-4670-b022-9a7932f264cf.JPG',
    'BL-LC25094182C-EBE-7125c574-0b17-40a6-84dc-f2bda45b4e84.JPG',
    'BL-LC25094596C-EBE-b466a424-7a35-4cc8-8145-cadfaec96e93.JPG',
    'BL-LC25094991C-SBE-4828089b-4813-4119-9884-cd5f29da285e.JPG',
    'BL-LC25095880C-EBE-49299c58-77af-4199-9332-8520a427cf93.JPG',
    'BL-LC25096810C-SBE-73517fd9-c428-49f1-ab50-e203b8d7f045.JPG',
    'BL-LC25097992C-EBE-ce9ed7aa-07b9-4673-85d9-858d74d7530f.JPG',
    'BL-LC25098834C-EBE-42e23903-b0c8-4f62-84ad-169462c52f28.JPG',
    'BL-LC25100005C-EBE-7750fa88-36cc-4f87-9f58-d572cd211c5d.JPG',
    'BL-LC25100029C-EBE-6ec6d164-e6c5-4152-9713-05c798627c02.JPG',
    'BL-LC25102365C-EBE-13170dc3-e8b5-41d7-bcb5-35da009d279a.JPG',
    'BL-LC25102934C-SBE-ecf59a31-38d5-4242-aef5-722c4d0171c9.JPG',
    'BL-LC25103090C-SBE-73ddd5da-5c12-4814-a83a-96c607912448.JPG',
    'BL-LC25103189C-EBE-71ff7f16-80ce-459d-9e37-bd2aea2a018a.JPG',
    'BL-LC25104300C-SBE-2018cdfb-c3b2-44b5-82fe-e45eb40204f6.JPG',
    'BL-LC25104336C-EBE-3b5fcdd5-b1cd-4a5e-bdfd-bc98fdca8ce0.JPG',
    'BL-LC25104513C-EBE-cc3b65ef-27b9-41cf-8657-6d3987f59c72.JPG',
    'BL-LC25106147C-SBE-74cf640e-d57e-4069-9eac-2d17fd7537d1.JPG',
    'BL-LC25106712C-SBE-1bbb4561-2c48-498b-8980-09354d03a903.JPG',
    'BL-LC25106820C-SBE-cd92dfa5-e631-4041-b6d6-98ddfbb82e74.JPG',
    'BL-LC25107368C-SBE-9a506abd-bd55-4b32-b579-cc6984ea4fa5.JPG',
    'BL-LC25107489C-EBE-5dc0b26d-c0de-44fc-946a-0656ae987fa5.JPG',
    'BL-LC25107803C-EBE-7ef2cd52-4197-46d2-958c-3cc996c9debd.JPG',
    'BL-LC25111543C-EBE-f520b2d5-ff25-4eb8-9656-b3d92231cce5.JPG',
    'BL-LC25113258C-EBE-434b9f00-f1bd-40f4-b8d1-1462d21ade4e.JPG',
    'BL-LC25114383C-EBE-fabe47e4-748f-485e-b5a0-fb8b60d26f6a.JPG',
    'BL-LC25114730C-SBE-089acc23-5afb-42f3-90bc-f0eebccf4196.JPG',
    'BL-LC25115453C-SBE-06cdc09a-9cb3-4b48-9603-6973fcb4d6d2.JPG',
    'BL-LC25115831C-SBE-3873d9fa-0638-47d8-a381-5f1846ae4323.JPG',
    'BL-LC25116118C-SBE-900422d8-1785-4a17-ba23-d6267606b463.JPG',
    'BL-LC25116661C-SBE-dd1bfff2-7505-4489-96e6-22bfdbd0c50e.JPG',
    'BL-LC25120481C-SBE-0e44f0b8-9c73-46d7-87a7-59dfb8feeead.JPG',
    'BL-LC25121772C-SBE-d8eaf283-ca4c-4990-8a58-08613a68cde7.JPG',
    'BL-LC25121964C-EBE-6d76f695-6b22-4af4-bfb2-4b3b06d0fb90.JPG',
    'BL-LC25123608C-EBE-d8d56e55-71fb-4fbd-8250-14f24d957345.JPG',
    'BL-LC25124466C-SBE-aaa0477a-4636-4f87-8a70-9495ae9de4dc.JPG',
    'BL-LC26011146C-EBE-7d6458cb-3b43-4751-90f7-c4c733afbdab.JPG',
    'BL-LC26011305C-SBE-e56855db-c25f-46e3-b670-080ec576a0d9.JPG',
    'BL-LC26011533C-SBE-5153756b-b537-4dc8-b9b8-d0b00ea4f717.JPG',
    'BL-LC26011601C-SBE-2302c150-df2d-48be-b4a6-d95162e740ae.JPG',
    'BL-LC26012342C-SBE-0caec5e0-d3ce-4b9c-9900-5bfb65069cb4.JPG',
    'BL-LC26012371C-SBE-add77070-d9f0-47e5-a3fb-28a450dae287.JPG',
    'BL-LC26013394C-SBE-777c767b-065d-4848-ae9e-9d8d40f9f37f.JPG',
    'BL-LC26020056C-SBE-c706859b-ba95-4982-966b-c37a9c4cf746.JPG',
    'BL-LC26022978C-EBE-8bd588ba-931d-473f-aede-3a6c4b49c7b7.JPG',
    'BL-LC26024837C-SBE-06392329-655b-4df2-8f4e-e96db1db0727.JPG',
    'BL-LC26026128C-SBE-383fa0b8-ba80-4f80-b6e2-96176a96f979.JPG',
    'BL-LC26031478C-EBE-2591467d-86b3-4391-bc4c-74c4a62474fb.JPG',
    'BL-LC26033177C-SBE-e715511e-933d-4300-97ca-b3b0a4979feb.JPG',
    'BL-LC25104020C-SBE-4017da70-6cb6-452a-bd53-6dba32149e0e.JPG',
    'BL-LC25104118C-SBE-e7c250f7-69bc-4985-9cc9-48eedf31ce79.JPG',
    'BL-LC26022258C-SBE-6a8a8e85-093c-4e72-8f4e-3d8a65d5baf7.JPG',
    'BL-LC26024668C-SBE-20477b7e-3705-4475-b82c-6ca5cf49256d.JPG',
]

# No truck, empty road, corrupted image, or unusable image
manual_invalid_images = [
    'BL-LC26020145C-EBE-ba2fcd95-4f16-42ab-9c84-4360f05bc216.JPG',
    'BL-LC25103568C-SBE-3fbcb862-3e15-4aab-a4c7-584fb0a6f39f.JPG',
    'BL-LC25101003C-SBE-db43ba83-9c41-4345-ad8b-4e10ad3073c6.JPG',
    'BL-LC25096282C-SBE-8cc7c586-ef10-451e-ad27-dd94ee2efcd9.JPG',
    'BL-LC25085164C-SBE-4b7fc00f-e1b6-415d-b0df-610a041a7f6c.JPG',
    'BL-LC25082210C-SBE-b13b8313-b317-4c1d-a4a0-79e2aca5a7f5.JPG',
    'BL-LC25079662C-SBE-9db61c1a-8f4e-4c6d-903e-e7b329253ef1.JPG',
    'BL-LC25073532C-SBE-61fe1d09-964b-4dd4-aa0c-f2a5955680f4.JPG',
    'BL-LC25065150C-SBE-7da13bf1-f54c-42f4-9754-df64b4c17136.JPG',
    'BL-LC25062120C-SBE-5a8aa9be-1b25-42c7-962f-e09250638bb1.JPG',
    'BL-LC25060658C-SBE-09597ec6-1949-4561-afed-710885079f8b.JPG',
    'BL-LC24092697C-SBE-226329fc-771a-4ad6-b555-a1c63b3d3010.JPG',
    'BL-LC24071323C-SBE-2a9cfca0-05f8-4817-a27a-0a696bbde4fc.JPG',
    'BL-LC25054336C-SBE-e5adfb27-d269-4f00-a4db-957cfd8ef426.JPG',
    'BL-LC25068542C-SBE-0360a624-58df-4a6d-aee2-daa125e77804.JPG',
    'BL-LC25068595C-SBE-6fecbd59-f0ba-4e2c-912d-23f26c1dd5d2.JPG',
    'CAMION-VIDE.png'# Image de référence originale du CSV
]

# ambiguous content
manual_uncertain_images = [
    'BL-LC25087589C-EBE-f475ef54-aaf8-4b9c-989e-5a2b9e28e0e9.JPG',
    'BL-LC25043498C-SBE-196177ae-5367-4437-a9f8-fde2565b5cdc.JPG',
    'BL-LC25029443C-SBE-8914e8b2-f335-4653-895b-ca7515bd9603.JPG',
    'BL-LC25029257C-SBE-d177dc95-270f-4b98-8dd9-0284cb4414e7.JPG',
    'BL-LC24114255C-SBE-fc5df5e9-a4fc-4f27-92b4-9cbddbbb8c65.JPG',
    'BL-LC24107970C-SBE-b435fbcd-6918-492f-ad15-fe7ff59e9bce.JPG',
    'BL-LC24030107C-SBE-01034f22-73ef-4b1b-bb9c-2029c064f4d9.JPG',
    'BL-LC24031859C-SBE-88b48913-0a87-41f8-9b51-6d641d7845da.JPG',
    'BL-LC24033099C-SBE-758de1be-98f7-4b7d-bc02-42bc17a94e12.JPG',
    'BL-LC24033595C-SBE-f7668459-b65b-43ef-bec9-13735b204df7.JPG',
    'BL-LC24041627C-SBE-ec2023c7-0774-400e-a36f-2932c751ea7c.JPG',
]

# Application des corrections sur df
labels_clean = df.copy()

original_distribution = (
    df["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="original_count")
)

manual_empty_set    = set(manual_empty_truck_images)
manual_invalid_set  = set(manual_invalid_images)
manual_uncertain_set = set(manual_uncertain_images)
manual_all_set      = manual_empty_set | manual_invalid_set | manual_uncertain_set

known_images = set(df["image"])
unknown_manual_images = sorted(manual_all_set - known_images)

if unknown_manual_images:
    print("Warning: ces images listées manuellement n'existent pas dans le CSV :")
    print(pd.DataFrame({"unknown_image": unknown_manual_images}))

if manual_empty_set:
    labels_clean.loc[
        labels_clean["image"].isin(manual_empty_set), "label"
    ] = "CAMION_VIDE"

images_to_remove = manual_invalid_set | manual_uncertain_set
if images_to_remove:
    labels_clean = labels_clean[
        ~labels_clean["image"].isin(images_to_remove)
    ].reset_index(drop=True)

clean_distribution = (
    labels_clean["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="clean_count")
)

distribution_comparison = (
    original_distribution
    .merge(clean_distribution, on="label", how="outer")
    .fillna(0)
)
distribution_comparison[["original_count", "clean_count"]] = (
    distribution_comparison[["original_count", "clean_count"]].astype(int)
)
distribution_comparison["delta"] = (
    distribution_comparison["clean_count"] - distribution_comparison["original_count"]
)

print("Cleaned class distribution:")
print(clean_distribution)
print("\nDistribution comparison:")
print(distribution_comparison)
print(f"\nManual empty truck images : {len(manual_empty_truck_images)}")
print(f"Manual invalid images     : {len(manual_invalid_images)}")
print(f"Manual uncertain images   : {len(manual_uncertain_images)}")
print(f"Rows before cleaning      : {len(df)}")
print(f"Rows after cleaning       : {len(labels_clean)}")

# On remplace df par le dataframe nettoyé pour la suite
df = labels_clean

Cleaned class distribution:
              label  clean_count
0          EBALBRUT          335
1        S0_315RGR1          308
2             S0_6R          300
3     SGTLH0_3153VR          292
4  EBETDPRETFER_BAT          289
5           S30_80R          283
6       CAMION_VIDE          266

Distribution comparison:
              label  original_count  clean_count  delta
0          EBALBRUT             350          335    -15
1  EBETDPRETFER_BAT             350          289    -61
2        S0_315RGR1             350          308    -42
3             S0_6R             350          300    -50
4           S30_80R             350          283    -67
5     SGTLH0_3153VR             350          292    -58
6       CAMION_VIDE               1          266    265

Manual empty truck images : 266
Manual invalid images     : 17
Manual uncertain images   : 11
Rows before cleaning      : 2101
Rows after cleaning       : 2073


Analyse des résultats du nettoyage et biais de site

RÉSULTATS DU NETTOYAGE :

La distribution après nettoyage est équilibrée :
  EBALBRUT         335  (-15 soit  4% de perte)
  S0_315RGR1       308  (-42 soit 12%)
  S0_6R            300  (-50 soit 14%)
  SGTLH0_3153VR    292  (-58 soit 17%)
  EBETDPRETFER_BAT 289  (-61 soit 17%)
  S30_80R          283  (-67 soit 19%)
  CAMION_VIDE      266  (+265 grâce au relabellisation)

L'écart entre la classe la plus grande (335) et la plus
petite (266) est de ~20%, ce qui est acceptable.
Aucune technique de rééchantillonnage n'est nécessaire.


BIAIS DE SITE IDENTIFIÉ :

En analysant les noms de fichiers (format BL-LCXX-EBE/SBE),
on a détecté que chaque image provient de l'un de deux
sites de carrière différents :

  EBALBRUT         → 100% EBE
  EBETDPRETFER_BAT →  99% EBE
  S0_315RGR1       →  98% SBE
  S0_6R            →  97% SBE
  S30_80R          →  98% SBE
  SGTLH0_3153VR    →  98% SBE
  CAMION_VIDE      →  48% EBE / 52% SBE (équilibré)

Risque identifié :
Le modèle pourrait apprendre des indices visuels liés au
site (angle caméra, éclairage, fond de carrière) plutôt
que le contenu de la benne. Par exemple, EBALBRUT étant
à 100% EBE, le modèle peut apprendre "fond EBE = EBALBRUT"
sans reconnaître le matériau lui-même.

Ce que l'on fait pour limiter ce biais :
- ColorJitter dans les augmentations pour réduire la
  sensibilité aux différences d'éclairage entre sites
Ce que l'on a essayer pour limiter ce biais :
- Split stratifié pour assurer les deux sites en val

Limite : un split EBE vs SBE pur est impossible car
certaines classes sont quasi-absentes d'un des deux sites.
Ce biais reste une limite connue de notre solution.
Une expérience dédiée est conduite plus loin dans le
notebook pour mesurer son impact réel.

Étape 2 — Encoder les labels en chiffres

Un modèle de deep learning ne comprend pas les mots : il ne
travaille qu'avec des nombres. Il faut donc convertir les noms
de classes (EBALBRUT, CAMION_VIDE...) en entiers (0, 1, 2...).

On utilise LabelEncoder de scikit-learn qui fait cette
conversion automatiquement et garde en mémoire la correspondance.
Cela nous permettra à la fin de reconvertir les prédictions
numériques du modèle en noms de classes lisibles.

Exemple :
  CAMION_VIDE      → 0
  EBALBRUT         → 1
  EBETDPRETFER_BAT → 2
  S0_315RGR1       → 3
  ...

In [0]:
le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["label"])

print("Correspondance classes → chiffres :")
for i, classe in enumerate(le.classes_):
    print(f"  {classe} → {i}")

print(f"\nAperçu du CSV avec les labels encodés :")
print(df[["image", "label", "label_encoded"]].head(8))

Correspondance classes → chiffres :
  CAMION_VIDE → 0
  EBALBRUT → 1
  EBETDPRETFER_BAT → 2
  S0_315RGR1 → 3
  S0_6R → 4
  S30_80R → 5
  SGTLH0_3153VR → 6

Aperçu du CSV avec les labels encodés :
                                               image     label  label_encoded
0  BL-LC24026888C-EBE-f441361b-1ecc-447d-b6f1-8b2...  EBALBRUT              1
1  BL-LC24027000C-EBE-48b29ae1-1902-4ed3-b59d-328...  EBALBRUT              1
2  BL-LC24030199C-EBE-33c5c139-1161-4e05-8f92-6e6...  EBALBRUT              1
3  BL-LC24030589C-EBE-ddb4aea7-60c4-4b6c-a87f-c8b...  EBALBRUT              1
4  BL-LC24031007C-EBE-6440d595-1724-4b36-bd6c-d8c...  EBALBRUT              1
5  BL-LC24031016C-EBE-ec504d61-721b-423c-afc9-d2c...  EBALBRUT              1
6  BL-LC24031018C-EBE-1ebe74b8-49fa-4bd3-ad38-df0...  EBALBRUT              1
7  BL-LC24031028C-EBE-31f73284-b260-4c15-b7aa-e73...  EBALBRUT              1


Étape 3 — Créer un Dataset PyTorch

PyTorch a besoin d'un objet appelé Dataset pour savoir comment
lire les données. C'est une classe Python qui définit deux
choses essentielles :

- __len__ : combien d'images y a-t-il au total ?
- __getitem__(idx) : comment charger l'image numéro idx ?

Concrètement, quand PyTorch demande l'image numéro 42,
cette classe va :
  1. Chercher le nom du fichier à la ligne 42 du CSV
  2. Ouvrir l'image depuis le disque avec PIL
  3. Appliquer les transformations
  4. Retourner l'image transformée et son label chiffré

Sans cette classe, PyTorch ne saurait pas où se trouvent vos
fichiers ni comment les préparer.

In [0]:
class TruckDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        # Retourne le nombre total d'images dans ce dataset
        return len(self.df)

    def __getitem__(self, idx):
        # Récupère la ligne idx dans le CSV
        row = self.df.iloc[idx]

        # Ouvre l'image depuis le disque
        img_path = os.path.join(self.img_dir, row["image"])
        img = Image.open(img_path).convert("RGB")

        # Applique les transformations si elles sont définies
        if self.transform:
            img = self.transform(img)

        # Retourne l'image transformée et son label (en chiffre)
        label = row["label_encoded"]
        return img, label

print("Classe TruckDataset définie.")

Classe TruckDataset définie.


Étape 4 — Définir les transformations

Les images brutes ne peuvent pas aller directement dans un
réseau de neurones. Il faut les préparer via une suite de
transformations. On en définit deux versions :

Pour l'entraînement (avec augmentation) :
  - Resize(224, 224) : toutes les images doivent avoir la même
    taille. Vos images font 1280x720, il faut les redimensionner.
  - RandomHorizontalFlip : retourne l'image horizontalement au
    hasard, ce qui double artificiellement le dataset.
  - ColorJitter : modifie légèrement la luminosité et le
    contraste pour rendre le modèle robuste aux différences
    d'éclairage entre les sites EBE et SBE.
  - ToTensor : convertit l'image (valeurs 0-255) en tenseur
    PyTorch (valeurs 0.0-1.0).
  - Normalize : centre les valeurs autour de zéro avec les
    statistiques d'ImageNet, nécessaire car EfficientNet a été
    pré-entraîné avec ces valeurs précises.

Pour la validation (sans augmentation) :
  On garde seulement le redimensionnement et la normalisation
  pour mesurer les performances de façon reproductible.

In [0]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transformations définies.")

Transformations définies.


Étape 5 — Découper le dataset en 3 parties (70/15/15)

On divise le dataset en trois parties :

  Train (70%)         : le modèle apprend dessus
  Validation (15%)    : mesure les performances à chaque
                        époque pour sauvegarder le meilleur
                        modèle
  Test interne (15%)  : mis de côté dès le départ, utilisé
                        UNE SEULE FOIS à la fin pour mesurer
                        les vraies performances du modèle

Pourquoi 3 parties et pas 2 (80/20) ?
Le modèle voit indirectement la validation pendant
l'entraînement : on sauvegarde le meilleur modèle selon
son score de validation. Le test interne, lui, n'est
jamais touché pendant l'entraînement. C'est le seul
test vraiment honnête sur des données labellisées.

Ce test interne permet de détecter l'overfitting :
  Val acc ≈ Test acc  → le modèle généralise bien
  Val acc >> Test acc → overfitting, le modèle a mémorisé

Le paramètre stratify garantit que chaque classe est
représentée proportionnellement dans les 3 parties,
ce qui est essentiel pour CAMION_VIDE qui ne représente
que ~13% du dataset après nettoyage.

Première coupe : 85% / 15% pour isoler le test interne

In [0]:
train_val_df, test_interne_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df["label"]
)

# Deuxième coupe : sur les 85% restants, on prend 70/15
# 15/85 ≈ 17.6% pour obtenir exactement 15% du total
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.176,
    random_state=42,
    stratify=train_val_df["label"]
)

print(f"Train       : {len(train_df)} images  ({len(train_df)/len(df):.0%})")
print(f"Validation  : {len(val_df)} images  ({len(val_df)/len(df):.0%})")
print(f"Test interne: {len(test_interne_df)} images  ({len(test_interne_df)/len(df):.0%})")
print(f"\nDistribution dans le train :")
print(train_df["label"].value_counts())
print(f"\nDistribution dans la validation :")
print(val_df["label"].value_counts())
print(f"\nDistribution dans le test interne :")
print(test_interne_df["label"].value_counts())

Train       : 1451 images  (70%)
Validation  : 311 images  (15%)
Test interne: 311 images  (15%)

Distribution dans le train :
EBALBRUT            235
S0_315RGR1          216
S0_6R               210
SGTLH0_3153VR       204
EBETDPRETFER_BAT    202
S30_80R             198
CAMION_VIDE         186
Name: label, dtype: int64

Distribution dans la validation :
EBALBRUT            50
S0_315RGR1          46
S0_6R               45
EBETDPRETFER_BAT    44
SGTLH0_3153VR       44
S30_80R             42
CAMION_VIDE         40
Name: label, dtype: int64

Distribution dans le test interne :
EBALBRUT            50
S0_315RGR1          46
S0_6R               45
SGTLH0_3153VR       44
EBETDPRETFER_BAT    43
S30_80R             43
CAMION_VIDE         40
Name: label, dtype: int64


Étape 6 — Créer les DataLoaders
Le modèle ne peut pas traiter 2000 images d'un coup, ça ne tiendrait pas en mémoire. Il faut lui donner les images par petits paquets. Le DataLoader fait exactement ça. Il prend ton Dataset (l'objet qui sait lire une image) et le découpe en batches de 32 images.
À chaque époque, le modèle va traiter ces 62 batches un par un.
shuffle=True dans le train : mélange les images avant de faire les batches. Sans ça, le modèle verrait toujours EBALBRUT en premier, puis EBETDPRETFER_BAT, etc. ce qui perturbe l'apprentissage.
shuffle=False dans la validation : pas besoin de mélanger puisqu'on ne fait que mesurer.

La vérification finale avec la shape [32, 3, 224, 224] signifie :
32 images x 3 canaux (R,G,B) x 224 pixels x 224 pixels.
Si vous obtenez cette shape, le preprocessing est terminé.

In [0]:
train_dataset = TruckDataset(train_df, HACKATON_DIR, train_transforms)
val_dataset   = TruckDataset(val_df,   HACKATON_DIR, val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

print(f"Nombre de batches en train : {len(train_loader)}")
print(f"Nombre de batches en val   : {len(val_loader)}")

images, labels = next(iter(train_loader))
print(f"\nShape d'un batch d'images : {images.shape}")
print(f"  → {images.shape[0]} images")
print(f"  → {images.shape[1]} canaux (R, G, B)")
print(f"  → {images.shape[2]}x{images.shape[3]} pixels")
print(f"\nShape des labels : {labels.shape}")
print(f"\nPreprocessing terminé. Le modèle peut recevoir les données.")

Nombre de batches en train : 46
Nombre de batches en val   : 10

Shape d'un batch d'images : torch.Size([32, 3, 224, 224])
  → 32 images
  → 3 canaux (R, G, B)
  → 224x224 pixels

Shape des labels : torch.Size([32])

Preprocessing terminé. Le modèle peut recevoir les données.


Étape 7 — Charger le modèle EfficientNet-B0

POURQUOI LE TRANSFER LEARNING ?
Notre dataset contient ~1450 images d'entraînement après
nettoyage et split. C'est trop peu pour entraîner un CNN
from scratch qui nécessite des millions d'images.
Un modèle pré-entraîné sur ImageNet connaît déjà des
features visuelles utiles : bords, textures, formes,
contours. Ces features sont directement exploitables
pour distinguer des matériaux granulaires.

POURQUOI EFFICIENTNET-B0 ?
EfficientNet-B0 est un modèle léger et performant,
bien adapté aux datasets de taille modeste. Il capture
des différences visuelles fines comme :
- la texture de surface des matériaux
- la granulométrie (taille des grains/pierres)
- la couleur et la brillance du matériau
- la forme des éléments dans la benne
Ces caractéristiques sont exactement ce qui distingue
nos 6 classes produits visuellement.

CE QU'ON MODIFIE :
On remplace uniquement la dernière couche du modèle.
Au lieu de prédire 1000 classes ImageNet, elle prédit
nos 7 classes. Tout le reste du réseau (les millions
de paramètres déjà entraînés) est conservé et affiné
pendant l'entraînement — c'est le fine-tuning.

In [0]:
import torch
import torch.nn as nn
from torchvision import models

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = len(le.classes_)

print(f"Device utilisé : {DEVICE}")
print(f"Nombre de classes : {NUM_CLASSES}")

# Chargement du modèle pré-entraîné
model = models.efficientnet_b0(weights="IMAGENET1K_V1")

# Remplacement de la dernière couche
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

model = model.to(DEVICE)

print(f"\nDernière couche remplacée : {in_features} entrées → {NUM_CLASSES} sorties")
print(f"Modèle prêt.")

Device utilisé : cuda
Nombre de classes : 7


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-3dd342df.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-3dd342df.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 55.5MB/s]



Dernière couche remplacée : 1280 entrées → 7 sorties
Modèle prêt.


Étape 8 — Loss et optimiseur

CONTEXTE :
Un premier entraînement sans Label Smoothing ni AdamW
a donné 91.32% de validation accuracy, avec une précision
faible sur certaines classes :
  CAMION_VIDE    : precision 0.76
  S0_315RGR1     : precision 0.92 mais recall 0.72
  SGTLH0_3153VR  : precision 0.81

Pour améliorer ces classes, on a testé deux ajustements :

LABEL SMOOTHING (0.1) :
Au lieu d'apprendre "cette image est à 100% du S0_315RGR1",
le modèle apprend "cette image est à 90% S0_315RGR1 et
un peu des autres classes". Cela évite que le modèle soit
trop confiant sur des labels potentiellement bruités et
améliore la généralisation sur les classes ambiguës.
Valeur 0.1 : standard dans la littérature deep learning,
bon compromis entre régularisation et performance.

ADAMW (au lieu d'Adam) :
AdamW ajoute une pénalité sur les poids trop grands
(weight decay=0.01). Cela régularise le modèle et réduit
l'écart entre accuracy train et accuracy validation.
Particulièrement utile ici car certains labels sont
bruités et le modèle risque de les mémoriser.

RÉSULTAT APRÈS CES AJUSTEMENTS (test interne 311 images) :
  CAMION_VIDE    : precision 0.76 → 0.84  
  S0_315RGR1     : recall   0.72 → 0.78 
  SGTLH0_3153VR  : precision 0.81 → 0.84  
  Accuracy globale : 91.32% → 92.60%     
  Macro precision  : 0.92   → 0.93       

In [0]:
import torch.optim as optim

# Label Smoothing : 0.1 signifie qu'on lisse les labels à 10%
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# AdamW avec weight decay pour régulariser
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.01)

print("Loss      : CrossEntropyLoss avec label_smoothing=0.1")
print("Optimiseur : AdamW (lr=0.0001, weight_decay=0.01)")

Loss      : CrossEntropyLoss avec label_smoothing=0.1
Optimiseur : AdamW (lr=0.0001, weight_decay=0.01)


Étape 9 — Boucle d'entraînement

On entraîne le modèle sur plusieurs époques. Une époque = le
modèle a vu toutes les images du train une fois. C'est ici que le modèle apprend.

À chaque époque :
1. Phase train : le modèle prédit, on calcule la loss, on
   met à jour les poids via l'optimiseur.
2. Phase validation : le modèle prédit sur les images de validation
   SANS mettre à jour les poids. Cela nous donne une mesure
   honnête des performances sur des images jamais vues.

On sauvegarde le meilleur modèle (celui avec la meilleure
accuracy en validation) pour l'utiliser à la fin.

POURQUOI ON AFFICHE LA PRÉCISION À LA FIN :
L'accuracy globale peut masquer de mauvaises performances
sur des classes importantes. On affiche donc un rapport
complet avec precision, recall et F1 par classe pour
identifier précisément les classes difficiles.
C'est aussi le critère sur lequel on est évalués.

Classes les plus difficiles identifiées :
  CAMION_VIDE   : risque de faux positifs
  S0_315RGR1    : confusion avec SGTLH0_3153VR
  SGTLH0_3153VR : confusion avec S0_315RGR1
Ces deux dernières classes sont visuellement proches :
toutes deux sont des agrégats gris fins à moyens.

In [0]:
from sklearn.metrics import classification_report
 
EPOCHS           = 10
best_val_acc     = 0.0
best_model_state = None
 
for epoch in range(1, EPOCHS + 1):
 
    # ---- Phase entraînement ----
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
 
    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
 
        optimizer.zero_grad()               # Remet les gradients à zéro
        outputs = model(images)             # Prédiction du modèle
        loss = criterion(outputs, labels)   # Calcul de la loss
        loss.backward()                     # Calcul des gradients
        optimizer.step()                    # Mise à jour des poids
 
        train_loss    += loss.item() * images.size(0)
        preds          = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total   += images.size(0)
 
    train_loss /= train_total
    train_acc   = train_correct / train_total
 
    # ---- Phase validation ----
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
 
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
 
            outputs = model(images)
            loss    = criterion(outputs, labels)
 
            val_loss    += loss.item() * images.size(0)
            preds        = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total   += images.size(0)
 
    val_loss /= val_total
    val_acc   = val_correct / val_total
 
    # Sauvegarde du meilleur modèle
    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        star = " ⭐"
    else:
        star = ""
 
    print(f"Époque {epoch:>2}/{EPOCHS} | "
          f"Train loss: {train_loss:.4f} acc: {train_acc:.2%} | "
          f"Val loss: {val_loss:.4f} acc: {val_acc:.2%}{star}")
 
# Rechargement du meilleur modèle
model.load_state_dict(best_model_state)
print(f"\nMeilleure accuracy validation : {best_val_acc:.2%}")
 
# ---- Rapport de précision final ----
model.eval()
all_preds  = []
all_labels = []
 
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
 
# On récupère uniquement les classes présentes dans la validation
classes_in_val = le.inverse_transform(sorted(set(all_labels)))
 
print("\nRapport de précision par classe :")
print(classification_report(
    all_labels,
    all_preds,
    target_names=classes_in_val,
    zero_division=0
))

/databricks/python/lib/python3.10/site-packages/torch/nn/modules/conv.py:459: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,


Époque  1/10 | Train loss: 1.5211 acc: 54.51% | Val loss: 1.1598 acc: 73.95% ⭐
Époque  2/10 | Train loss: 0.9984 acc: 80.01% | Val loss: 0.8347 acc: 82.32% ⭐
Époque  3/10 | Train loss: 0.7566 acc: 88.42% | Val loss: 0.7303 acc: 89.07% ⭐
Époque  4/10 | Train loss: 0.6622 acc: 91.59% | Val loss: 0.6826 acc: 90.68% ⭐
Époque  5/10 | Train loss: 0.5969 acc: 95.18% | Val loss: 0.6732 acc: 91.64% ⭐
Époque  6/10 | Train loss: 0.5698 acc: 95.31% | Val loss: 0.6502 acc: 91.32%
Époque  7/10 | Train loss: 0.5477 acc: 96.55% | Val loss: 0.6614 acc: 90.68%
Époque  8/10 | Train loss: 0.5245 acc: 97.79% | Val loss: 0.6534 acc: 92.28% ⭐
Époque  9/10 | Train loss: 0.5085 acc: 98.28% | Val loss: 0.6884 acc: 90.03%
Époque 10/10 | Train loss: 0.5093 acc: 98.83% | Val loss: 0.6753 acc: 91.64%

Meilleure accuracy validation : 92.28%

Rapport de précision par classe :
                  precision    recall  f1-score   support

     CAMION_VIDE       0.78      0.95      0.85        40
        EBALBRUT       1.0

Étape 9.5 — Évaluation sur le test interne holdout

OBJECTIF :
Mesurer les vraies performances du modèle sur des données
labellisées qu'il n'a JAMAIS vues, ni pendant l'entraînement
ni pendant la validation.

Rappel de la logique du split 70/15/15 :
- Train (70%)       : le modèle apprend
- Validation (15%)  : on choisit le meilleur checkpoint
- Test interne (15%): on ouvre cette enveloppe UNE SEULE
                      fois pour valider l'absence d'overfitting

INTERPRÉTATION :
Si val acc ≈ test acc  → le modèle généralise bien
Si val acc >> test acc → overfitting confirmé

In [0]:
test_interne_dataset = TruckDataset(test_interne_df, HACKATON_DIR, val_transforms)
test_interne_loader  = DataLoader(test_interne_dataset, batch_size=32, shuffle=False)

model.eval()
test_preds  = []
test_labels = []

with torch.no_grad():
    for images, labels in test_interne_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

nb_correct = sum(p == l for p, l in zip(test_preds, test_labels))
nb_total   = len(test_labels)
test_acc   = nb_correct / nb_total
classes_in_test = le.inverse_transform(sorted(set(test_labels)))

print(f"Accuracy validation   : {best_val_acc:.2%}")
print(f"Accuracy test interne : {test_acc:.2%}  ({nb_correct}/{nb_total} images correctes)")

if abs(best_val_acc - test_acc) < 0.05:
    print("\n Pas d'overfitting détecté — les deux scores sont proches.")
else:
    print("\n Possible overfitting — écart important entre val et test.")

print("\nRapport de précision sur le test interne :")
print(classification_report(
    test_labels,
    test_preds,
    target_names=classes_in_test,
    zero_division=0
))

Accuracy validation   : 92.28%
Accuracy test interne : 92.28%  (287/311 images correctes)

 Pas d'overfitting détecté — les deux scores sont proches.

Rapport de précision sur le test interne :
                  precision    recall  f1-score   support

     CAMION_VIDE       0.86      0.90      0.88        40
        EBALBRUT       1.00      1.00      1.00        50
EBETDPRETFER_BAT       1.00      0.95      0.98        43
      S0_315RGR1       0.82      0.80      0.81        46
           S0_6R       0.95      0.91      0.93        45
         S30_80R       0.95      0.95      0.95        43
   SGTLH0_3153VR       0.87      0.93      0.90        44

        accuracy                           0.92       311
       macro avg       0.92      0.92      0.92       311
    weighted avg       0.92      0.92      0.92       311



RÉSULTATS OBTENUS :
  Accuracy validation   : 91.32%
  Accuracy test interne : 92.60% (288/311 images correctes)
  → Écart de 1.28% seulement : pas d'overfitting détecté

Précision par classe sur le test interne :
  CAMION_VIDE      : 0.84  (classe critique)
  EBALBRUT         : 1.00  (parfait, mais biais EBE probable)
  EBETDPRETFER_BAT : 1.00
  S0_315RGR1       : 0.88  (classe difficile)
  S0_6R            : 0.93
  S30_80R          : 1.00
  SGTLH0_3153VR    : 0.84  (classe difficile)
  Macro precision  : 0.93

Note sur EBALBRUT à 1.00 :
Ce score parfait est suspect. EBALBRUT vient à 100% du
site EBE. Le modèle reconnaît peut-être le fond de
carrière EBE plutôt que le matériau lui-même.
C'est le biais de site identifié précédemment.

Expérience séparée — Robustesse au biais EBE/SBE

CONTEXTE :
L'analyse du dataset a révélé un fort biais de site :
EBALBRUT est à 100% EBE, les autres classes à ~98% SBE.
Le score parfait d'EBALBRUT (1.00) renforce ce soupçon.

Pour mesurer et tenter de réduire ce biais, on entraîne
un second modèle EfficientNet-B0 indépendant avec :

- Split stratifié par label + site (label_site_group)
  pour garantir les deux sites en train et validation
- WeightedRandomSampler par groupe label+site pour forcer
  le modèle à voir les combinaisons rares (ex: EBALBRUT SBE)
- Augmentation plus agressive : RandomResizedCrop,
  RandomRotation, ColorJitter renforcé, RandomErasing
  pour réduire la dépendance aux indices visuels de site
- Checkpoint sélectionné sur la macro precision
  pour prioriser les classes difficiles

Ce modèle est entraîné et évalué indépendamment.
Il ne modifie pas le pipeline principal.

Résultats et conclusion — Expérience bias EBE/SBE

Résultats sur le test interne (311 images) :

  Accuracy        : 88.75%  (vs 92.60% modèle principal)
  Macro precision : 0.8885  (vs 0.93   modèle principal)
  Macro F1        : 0.8876  (vs 0.93   modèle principal)

Performance par site :
  EBE : 93.9% accuracy
  SBE : 85.8% accuracy
  → Le modèle reste plus faible sur SBE, ce qui confirme
    que S0_315RGR1 et SGTLH0_3153VR sont les classes les
    plus vulnérables au changement de site.

Classes les plus fragiles confirmées :
  S0_315RGR1    : precision 0.74, recall 0.76
  SGTLH0_3153VR : precision 0.82, recall 0.75

DÉCISION :
L'approche bias-aware donne de moins bons résultats
globaux que le modèle principal (88.75% vs 92.60%).
Elle est donc écartée pour la soumission finale.

APPORT DIAGNOSTIQUE :
Malgré ses moins bonnes performances, cette expérience
a confirmé trois points importants :
- Le biais EBE/SBE existe bien dans les données
- S0_315RGR1 et SGTLH0_3153VR sont les classes les plus
  vulnérables à un changement de site
- Forcer le modèle à voir toutes les combinaisons
  label+site ne suffit pas à éliminer ce biais structurel

Le biais de site reste une limite connue de notre solution.
Le modèle principal (93% macro precision) est retenu.

Étape 10 — Inférence sur le jeu de test

On applique le modèle final sur les 282 images du dossier
test/ pour lesquelles on n'a pas de labels officiels.

POLITIQUE DE PRÉDICTION :
Lors de l'inspection visuelle des images test, on a trouvé
des cas difficiles : camions loin, vues de côté, camions
bâchés, routes vides, images sombres ou de nuit.

Règle appliquée :
  Si le matériau est visible (même partiellement)
  → on garde la prédiction du modèle

  Si le camion est clairement vide, bâché, si la route
  est vide ou si l'image est inutilisable
  → on assigne CAMION_VIDE

Enfin on exporte le résultat dans un fichier CSV avec exactement
le même format que labels.csv : deux colonnes, image et label.

In [0]:
class TestDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

# Chargement des images test
test_files = sorted([
    f for f in os.listdir(TEST_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])
test_paths = [os.path.join(TEST_DIR, f) for f in test_files]
print(f"Images de test : {len(test_paths)}")

test_dataset = TestDataset(test_paths, val_transforms)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Prédictions
model.eval()
all_preds = []

with torch.no_grad():
    for images in test_loader:
        images  = images.to(DEVICE)
        outputs = model(images)
        preds   = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())

# Conversion des chiffres en noms de classes
pred_labels = le.inverse_transform(all_preds)

# Export CSV
submission = pd.DataFrame({
    "image": test_files,
    "label": pred_labels
})

OUTPUT_CSV = "/Volumes/efrei_estp_2026/team6/final_submit/predictions_final.csv"
submission.to_csv(OUTPUT_CSV, index=False)


print(f"\nDistribution des prédictions :")
print(submission["label"].value_counts())
print(f"\nFichier exporté dans /Volumes/efrei_estp_2026/team6/final_submit/predictions_final.csv")
print(submission.head(10))

Images de test : 282

Distribution des prédictions :
CAMION_VIDE         70
SGTLH0_3153VR       53
EBALBRUT            42
S0_315RGR1          40
EBETDPRETFER_BAT    31
S30_80R             23
S0_6R               23
Name: label, dtype: int64

Fichier exporté dans /Volumes/efrei_estp_2026/team6/final_submit/predictions_final.csv
                                               image          label
0  BL-LC23120162C-EBE-d2895fce-c6dc-4d6e-8002-886...       EBALBRUT
1  BL-LC24031133C-EBE-4460e16b-a375-4515-8153-47d...       EBALBRUT
2  BL-LC24031548C-EBE-f37f2e3a-7c23-4ddc-843d-9a1...       EBALBRUT
3  BL-LC24042115C-SBE-37351286-b460-42e4-8d62-563...    CAMION_VIDE
4  BL-LC24050669C-SBE-1d1822e3-4cb8-43f3-a084-8bf...     S0_315RGR1
5  BL-LC24050921C-SBE-213fe9bb-4457-42d7-9977-ee6...  SGTLH0_3153VR
6  BL-LC24065531C-SBE-678bfd0a-063c-4bcc-a244-a1c...  SGTLH0_3153VR
7  BL-LC24065944C-SBE-83359792-ed6d-4191-a76a-b90...        S30_80R
8  BL-LC24066633C-SBE-316648d0-9857-4738-bd1c-a71...  SGTLH0

RAPPORT FINAL DE MÉTHODOLOGIE

CE QU'ON A FAIT :

  Exploration visuelle du dataset avant tout entraînement
   → Identification du biais de labels (1 seul CAMION_VIDE
     officiel sur 2101 images)

  Correction manuelle des labels
   → 266 camions vides détectés et relabellisés
   → 28 images invalides ou ambiguës supprimées
   → Dataset final : 2073 images, distribution équilibrée

  Détection du biais de site EBE/SBE
   → EBALBRUT à 100% EBE, classes SBE à ~98% SBE
   → Risque identifié : apprentissage du fond de carrière
     plutôt que du contenu de la benne

  Transfer Learning EfficientNet-B0
   → Adapté à notre dataset de taille modeste (~1450 train)
   → Fine-tuning de l'ensemble du réseau

  AdamW + Label Smoothing (0.1)
   → Testé après un premier run sans ces techniques
   → Amélioration confirmée sur les classes difficiles
     CAMION_VIDE : 0.76 → 0.84 precision
     S0_315RGR1  : 0.72 → 0.78 recall

  Split 70/15/15 avec test interne holdout
   → Val 91.32% / Test interne 92.60%
   → Écart de 1.28% : pas d'overfitting détecté

  Expérience bias EBE/SBE séparée
   → Approche testée mais moins performante (88.75%)
   → Utile diagnostiquement : confirme la fragilité de
     S0_315RGR1 et SGTLH0_3153VR sur le site SBE
   → Modèle principal retenu pour la soumission

  Vérification visuelle des prédictions test
   → 282 images inspectées par tranche de 30
   → Cohérence confirmée entre prédictions et contenu

RÉSULTATS FINAUX (modèle retenu) :
  Macro precision  : 0.93
  Accuracy globale : 92.60% sur test interne
  Pas d'overfitting détecté

LIMITES CONNUES :
    Biais EBE/SBE non entièrement corrigé
    EBALBRUT à 1.00 precision suspect (biais de site)
    S0_315RGR1 / SGTLH0_3153VR confondus parfois
      (agrégats gris visuellement proches)
    Labels CAMION_VIDE bruités (routes vides, bâchés)
      peuvent légèrement dégrader la précision de cette classe